# CSCE 580 – Quiz 1
## Q3: Data analysis for social impact (Fire station data)

**What this notebook does:**
- Loads the provided CSV (8 columns, ~2,200 rows)
- Diagnoses data issues, missingness, type normalization
- Computes key metrics (resolution time, avg units per alarm, busiest shift)
- Builds a day-of-week × hour matrix with totals
- Runs two clustering methods and compares silhouette scores

> Download the dataset from Google Drive and set `data_path` below.


In [1]:
# Path setup — use repo structure (run from Quiz1/code/)
data_path = '../data/irmo.csv'
out_dir = '../data'


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

Path(out_dir).mkdir(parents=True, exist_ok=True)

# Flexible datetime parser
def to_dt(s):
    return pd.to_datetime(s, errors='coerce', utc=False)

df_raw = pd.read_csv(data_path)
df = df_raw.copy()
print('Shape:', df.shape)
df.head()


Shape: (2200, 8)


,XREF ID,DISPATCH UNIT,DISPATCH CREATED DATE,INCIDENT NUMBER,1ST UNIT ON SCENE,ALARM DATE TIME,CALL COMPLETE,SHIFT
0,2025107105,"BAT111, E171, LDR175",3/24/25 15:54,25-1368,BC-111,3/24/25 15:46,9/5/25 16:20,C
1,2025107223,BAT111,3/24/25 17:28,25-1369,BC-111,3/24/25 17:23,9/5/25 17:55,C
2,2025107415,E171,3/24/25 21:03,25-1370,E-171,3/24/25 21:02,9/4/25 21:09,C
3,2025107411,E171,3/24/25 21:03,25-1371,NaN,3/24/25 20:58,9/4/25 21:02,C
4,2025107384,"BAT111, E171, LDR175",3/24/25 21:43,25-1374,BC-111,3/24/25 20:30,9/4/25 21:36,C


### (a) Data issues

In [3]:
# --- Robust datetime parsing helpers ---
import re
import numpy as np
import pandas as pd

LIKELY_DT_COL = re.compile(r"(date|time|created|closed|opened|reported|updated|start|end)", re.I)

CANDIDATE_FORMATS = [
    "%Y-%m-%d",
    "%Y-%m-%d %H:%M",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%S%z",
    "%m/%d/%Y",
    "%m/%d/%Y %H:%M",
    "%m/%d/%Y %I:%M %p",
    "%d/%m/%Y",            # if your data uses day-first
    "%d/%m/%Y %H:%M",
]

def parse_dt_series(s: pd.Series, dayfirst=False) -> pd.Series:
    s = s.copy()

    # Already datetime?
    if pd.api.types.is_datetime64_any_dtype(s):
        return s

    # Numeric epochs/serials?
    if pd.api.types.is_numeric_dtype(s):
        a = s.astype("float64")
        # Heuristics: large numbers -> epoch ms, smaller -> seconds; Excel serials ~ 20k-60k
        out = pd.to_datetime(a, unit="ms", errors="coerce")
        bad = out.isna()
        if bad.any():
            out.loc[bad] = pd.to_datetime(a[bad], unit="s", errors="coerce")
        bad = out.isna()
        # Excel serial dates (days since 1899-12-30)
        maybe_excel = (a > 20000) & (a < 60000)
        if bad.any() and maybe_excel.any():
            excel = pd.to_datetime("1899-12-30") + pd.to_timedelta(a, unit="D")
            out.loc[bad & maybe_excel] = excel[bad & maybe_excel]
        return out

    # Strings -> try fast vectorized parse with known formats
    s_str = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # Try each explicit format
    mask_left = s_str.notna() & (s_str != "")
    for fmt in CANDIDATE_FORMATS:
        try:
            parsed = pd.to_datetime(s_str.where(mask_left), format=fmt, errors="coerce", dayfirst=dayfirst)
            fill = out.isna() & parsed.notna()
            out.loc[fill] = parsed.loc[fill]
            mask_left = mask_left & out.isna()
            if not mask_left.any():
                break
        except Exception:
            pass

    # Fallback: mixed/loose parsing (slower but OK as last resort)
    if out.isna().any():
        try:
            # Pandas >=2.0 supports format="mixed"
            parsed = pd.to_datetime(s_str.where(out.isna()), errors="coerce", dayfirst=dayfirst, format="mixed")
        except TypeError:
            parsed = pd.to_datetime(s_str.where(out.isna()), errors="coerce", dayfirst=dayfirst)
        out.loc[out.isna()] = parsed

    return out

# --- Apply to your frame ---
# 1) detect columns
date_cols = [c for c in df.columns if LIKELY_DT_COL.search(c)]
print("Detected date-like columns:", date_cols)

# 2) parse each into a *_dt column + small report
report = []
for c in date_cols:
    dtc = c + "_dt"
    df[dtc] = parse_dt_series(df[c], dayfirst=False)  # toggle to True if your data is dd/mm/yyyy
    ok = df[dtc].notna().mean()
    report.append({
        "column": c,
        "parsed_col": dtc,
        "parse_success_pct": round(100*ok, 2),
        "min": df[dtc].min(),
        "max": df[dtc].max(),
        "example": df[c].dropna().astype(str).head(1).tolist()[0] if df[c].notna().any() else None,
    })

rep = pd.DataFrame(report).sort_values("parse_success_pct", ascending=False)
display(rep)

# 3) global range from parsed cols only
dt_cols = [c for c in df.columns if c.endswith("_dt")]
if dt_cols:
    all_dt = pd.concat([df[c] for c in dt_cols], axis=0)
    min_dt, max_dt = all_dt.min(), all_dt.max()
    print(f"Data range (min → max): {min_dt} → {max_dt}")
else:
    print("No parsed datetime columns found.")

# 4) % missing (save)
missing = df.isna().mean().sort_values(ascending=False).to_frame("missing_frac")
missing.to_csv(f"{out_dir}/a2_missing_by_column.csv")
missing.head(10)


Detected date-like columns: ['DISPATCH CREATED DATE', 'ALARM DATE TIME']


,column,parsed_col,parse_success_pct,min,max,example
0,DISPATCH CREATED DATE,DISPATCH CREATED DATE_dt,100.00,2025-03-24 15:54:00,2025-08-31 23:03:00,3/24/25 15:54
1,ALARM DATE TIME,ALARM DATE TIME_dt,98.59,2025-03-24 15:46:00,2025-08-31 22:58:00,3/24/25 15:46


Data range (min → max): 2025-03-24 15:46:00 → 2025-08-31 23:03:00


,missing_frac
1ST UNIT ON SCENE,0.194545
SHIFT,0.031364
ALARM DATE TIME_dt,0.014091
ALARM DATE TIME,0.014091
CALL COMPLETE,0.014091
XREF ID,0.000000
DISPATCH UNIT,0.000000
INCIDENT NUMBER,0.000000
DISPATCH CREATED DATE,0.000000
DISPATCH CREATED DATE_dt,0.000000


In [4]:
# 3) Data issues: type drift, categorical normalization
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'example': df.apply(lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else None)
})
summary.to_csv(f"{out_dir}/a3_column_summary.csv")
summary


,dtype,n_unique,example
XREF ID,int64,2200,2025107105
DISPATCH UNIT,object,53,"BAT111, E171, LDR175"
DISPATCH CREATED DATE,object,2133,3/24/25 15:54
INCIDENT NUMBER,object,2199,25-1368
1ST UNIT ON SCENE,object,10,BC-111
ALARM DATE TIME,object,2158,3/24/25 15:46
CALL COMPLETE,object,1078,9/5/25 16:20
SHIFT,object,3,C
DISPATCH CREATED DATE_dt,datetime64[ns],2133,2025-03-24 15:54:00
ALARM DATE TIME_dt,datetime64[ns],2158,2025-03-24 15:46:00


In [5]:
# 4) Resolve data issues
clean = df.copy()

# Standardize likely categorical columns
for c in clean.columns:
    if clean[c].dtype == object:
        clean[c] = clean[c].astype(str).str.strip()

# Build canonical datetime columns if present
created_col = next((c for c in clean.columns if c.lower().startswith('created')), None)
closed_col  = next((c for c in clean.columns if c.lower().startswith('closed')), None)
if created_col:
    clean['created_dt'] = to_dt(clean[created_col])
if closed_col:
    clean['closed_dt'] = to_dt(clean[closed_col])

# Simple ID assignment
clean['CaseID'] = np.arange(1, len(clean)+1)

# Impute missing: numeric→median, categorical→'Unknown'
for c in clean.columns:
    if pd.api.types.is_numeric_dtype(clean[c]):
        med = clean[c].median()
        clean[c] = clean[c].fillna(med)
    else:
        clean[c] = clean[c].fillna('Unknown')

clean.to_csv(f"{out_dir}/a4_cleaned.csv", index=False)
print('Saved cleaned data.')
clean.head()


Saved cleaned data.


,XREF ID,DISPATCH UNIT,DISPATCH CREATED DATE,INCIDENT NUMBER,1ST UNIT ON SCENE,ALARM DATE TIME,CALL COMPLETE,SHIFT,DISPATCH CREATED DATE_dt,ALARM DATE TIME_dt,CaseID
0,2025107105,"BAT111, E171, LDR175",3/24/25 15:54,25-1368,BC-111,3/24/25 15:46,9/5/25 16:20,C,2025-03-24 15:54:00,2025-03-24 15:46:00,1
1,2025107223,BAT111,3/24/25 17:28,25-1369,BC-111,3/24/25 17:23,9/5/25 17:55,C,2025-03-24 17:28:00,2025-03-24 17:23:00,2
2,2025107415,E171,3/24/25 21:03,25-1370,E-171,3/24/25 21:02,9/4/25 21:09,C,2025-03-24 21:03:00,2025-03-24 21:02:00,3
3,2025107411,E171,3/24/25 21:03,25-1371,nan,3/24/25 20:58,9/4/25 21:02,C,2025-03-24 21:03:00,2025-03-24 20:58:00,4
4,2025107384,"BAT111, E171, LDR175",3/24/25 21:43,25-1374,BC-111,3/24/25 20:30,9/4/25 21:36,C,2025-03-24 21:43:00,2025-03-24 20:30:00,5


### (b) Exploratory analysis – alarms

In [6]:
# Helper: resolution time in minutes if times available
if 'created_dt' in clean and 'closed_dt' in clean:
    clean['resolution_min'] = (clean['closed_dt'] - clean['created_dt']).dt.total_seconds() / 60.0
else:
    clean['resolution_min'] = np.nan

# 1) Average resolution time (minutes)
avg_resolution = clean['resolution_min'].mean()
print('Average resolution time (min):', round(avg_resolution, 2))

# 2) Avg # units dispatched (try to guess a column)
unit_cols = [c for c in clean.columns if 'unit' in c.lower()]
if unit_cols:
    units_col = unit_cols[0]
    avg_units = pd.to_numeric(clean[units_col], errors='coerce').mean()
else:
    units_col = None
    avg_units = np.nan
print('Units column:', units_col)
print('Average units per alarm:', round(avg_units, 2) if pd.notna(avg_units) else 'N/A')

# 3) Busiest shift among A/B/C (infer column named like "Shift")
shift_col = next((c for c in clean.columns if 'shift' in c.lower()), None)
if shift_col:
    shift_counts = clean[shift_col].value_counts().reindex(['A','B','C']).fillna(0)
    busiest_shift = shift_counts.idxmax()
else:
    busiest_shift, shift_counts = 'N/A', pd.Series(dtype=int)
print('Busiest shift:', busiest_shift)
shift_counts


Average resolution time (min): nan
Units column: DISPATCH UNIT
Average units per alarm: N/A
Busiest shift: A


SHIFT
A    735
B    677
C    719
Name: count, dtype: int64

In [7]:
# 4) Day-of-week × hour matrix (with totals)
# Derive DoW and hour from created_dt if available
if 'created_dt' in clean:
    clean['dow'] = clean['created_dt'].dt.day_name()
    clean['hour'] = clean['created_dt'].dt.hour
else:
    # fallback: try any date-like column we created earlier
    dt_cols = [c for c in clean.columns if c.endswith('_dt')]
    use = dt_cols[0] if dt_cols else None
    if use:
        clean['dow'] = clean[use].dt.day_name()
        clean['hour'] = clean[use].dt.hour

if 'dow' in clean and 'hour' in clean:
    mat = pd.pivot_table(clean, index='hour', columns='dow', values='CaseID', aggfunc='count', fill_value=0)
    # Reorder days Monday→Sunday
    order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    mat = mat[[c for c in order if c in mat.columns]]
    mat['Total'] = mat.sum(axis=1)
    mat.loc['Total'] = mat.sum(axis=0)
    mat.to_csv(f"{out_dir}/b4_dow_hour_matrix.csv")
    display(mat)
else:
    print('Could not construct day-of-week/hour matrix — no datetime available.')


dow,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,Total
hour,,,,,,,,
0,8,8,6,2,3,7,4,38
1,11,8,7,8,3,8,10,55
2,4,5,3,3,4,8,8,35
3,9,10,8,1,9,4,9,50
4,4,4,4,2,7,6,4,31
5,9,3,8,4,6,4,5,39
6,4,5,7,7,10,6,10,49
7,15,15,8,11,14,5,10,78
8,13,13,18,8,8,7,14,81


### (c) Unsupervised learning – two methods & cluster quality

In [8]:
# Choose numeric features for clustering (auto-detect)
num_df = clean.select_dtypes(include=['number']).copy()
num_df = num_df.drop(columns=[c for c in num_df.columns if num_df[c].nunique() <= 1], errors='ignore')
num_df = num_df.dropna(axis=1, how='any')  # keep simple
print('Using numeric features:', list(num_df.columns)[:10], '...')

# Normalize features
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(num_df.values)

# Method 1: KMeans
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
labels_k = kmeans.fit_predict(X)
sil_k = silhouette_score(X, labels_k) if len(set(labels_k)) > 1 else np.nan
print(f'KMeans: k={k}, silhouette={sil_k:.4f}')

# Method 2: Agglomerative
agg = AgglomerativeClustering(n_clusters=k)
labels_a = agg.fit_predict(X)
sil_a = silhouette_score(X, labels_a) if len(set(labels_a)) > 1 else np.nan
print(f'Agglomerative: k={k}, silhouette={sil_a:.4f}')

# Pick better
better = 'KMeans' if (sil_k >= sil_a) else 'Agglomerative'
print('Better method (by silhouette):', better)

# Attach best labels and export
clean['cluster_best'] = labels_k if better=='KMeans' else labels_a
clean[['CaseID','cluster_best']].to_csv(f"{out_dir}/c_clusters.csv", index=False)


Using numeric features: ['XREF ID', 'CaseID', 'hour'] ...
KMeans: k=3, silhouette=0.3846


Agglomerative: k=3, silhouette=0.3569
Better method (by silhouette): KMeans


> **Interpretation (fill-in):**
- Inspect cluster centers (if KMeans) or summary stats by cluster to label them (e.g., short vs long incidents, high unit count vs low, etc.).

In [9]:
# --- Cluster profiling: numeric, datetime, and categorical --- 
# Requirements: a DataFrame named `clean` and a column `cluster_best` with cluster labels.

import re
import pandas as pd
from pathlib import Path

# ==== Config / output dir ====
out_dir = Path(out_dir) if 'out_dir' in globals() else Path("./outputs_q3")
out_dir.mkdir(parents=True, exist_ok=True)

# ==== Sanity checks ====
if 'clean' not in globals() or not isinstance(clean, pd.DataFrame):
    raise RuntimeError("Expected a DataFrame named `clean` to exist.")
if 'cluster_best' not in clean.columns:
    raise RuntimeError("`clean` must contain a 'cluster_best' column with cluster labels.")

# ==== Ensure numeric 'resolution_min' and (optional) numeric units count ====
# Try to derive units_count from a text-like "units" column if present
units_col_guess = next((c for c in clean.columns if 'unit' in c.lower()), None)
if units_col_guess and 'units_count' not in clean.columns:
    def count_units(val):
        if pd.isna(val):
            return 0
        toks = re.split(r'[,\s;/]+', str(val).strip())
        toks = [t for t in toks if t and t.upper() not in {'NONE'}]
        return len(toks)
    clean['units_count'] = clean[units_col_guess].apply(count_units)

# ==== NUMERIC aggregation (mean/median/count) ====
num_cols_all = clean.select_dtypes(include=['number']).columns.tolist()
# exclude the label column if it got cast to numeric (rare but safe)
num_cols = [c for c in num_cols_all if c != 'cluster_best']

prof_num = pd.DataFrame()
if num_cols:
    prof_num = clean.groupby('cluster_best')[num_cols].agg(['mean', 'median', 'count'])
    prof_num.to_csv(out_dir / "c_cluster_profile_numeric.csv")
    print(f"Saved -> {out_dir / 'c_cluster_profile_numeric.csv'}")
else:
    print("[Info] No numeric columns found to aggregate.")

# ==== DATETIME aggregation (min/max) ====
dt_cols = [c for c in clean.columns if pd.api.types.is_datetime64_any_dtype(clean[c])]
if dt_cols:
    prof_dt = clean.groupby('cluster_best')[dt_cols].agg(['min', 'max'])
    prof_dt.to_csv(out_dir / "c_cluster_profile_datetimes.csv")
    print(f"Saved -> {out_dir / 'c_cluster_profile_datetimes.csv'}")
else:
    prof_dt = pd.DataFrame()
    print("[Info] No datetime columns found to aggregate.")

# ==== CATEGORICAL/TEXT aggregation (nunique + top3 most frequent) ====
cat_cols = clean.select_dtypes(include=['object', 'category']).columns.tolist()

def top3(s: pd.Series) -> str:
    vc = s.value_counts(dropna=True).head(3)
    return "; ".join([f"{idx}:{cnt}" for idx, cnt in vc.items()])

prof_cat = pd.DataFrame()
if cat_cols:
    prof_cat = clean.groupby('cluster_best')[cat_cols].agg(['nunique', top3])
    prof_cat.to_csv(out_dir / "c_cluster_profile_categoricals.csv")
    print(f"Saved -> {out_dir / 'c_cluster_profile_categoricals.csv'}")
else:
    print("[Info] No categorical/text columns found to aggregate.")

# ==== Optional: one Excel workbook with three sheets ====
# Use xlsxwriter if available; fall back to default engine otherwise.
try:
    import xlsxwriter  # noqa: F401
    excel_kwargs = {"engine": "xlsxwriter"}
except Exception:
    excel_kwargs = {}

sheets = {}
if not prof_num.empty: sheets["numeric"] = prof_num
if dt_cols and not prof_dt.empty: sheets["datetimes"] = prof_dt
if cat_cols and not prof_cat.empty: sheets["categoricals"] = prof_cat

if sheets:
    with pd.ExcelWriter(out_dir / "c_cluster_profile.xlsx", **excel_kwargs) as xw:
        for name, df_ in sheets.items():
            df_.to_excel(xw, sheet_name=name, index=True)
    print(f"Saved -> {out_dir / 'c_cluster_profile.xlsx'}")
else:
    print("[Info] No sheets to write to Excel (no profiles were created).")

print("Done: numeric/datetime/categorical cluster profiles generated.")


Saved -> ../data/c_cluster_profile_numeric.csv
Saved -> ../data/c_cluster_profile_datetimes.csv
Saved -> ../data/c_cluster_profile_categoricals.csv
Saved -> ../data/c_cluster_profile.xlsx
Done: numeric/datetime/categorical cluster profiles generated.
